# Phase 1 One-Click Automated Pipeline (Google Colab T4)

**Expected Total Runtime**: ~45–60 minutes.

**Disconnect & Resumption Policy**: All steps in this notebook are fully idempotent and resumable.
If your Google Colab session disconnects or resets at any point, simply select **Runtime -> Run all**.
Training automatically resumes from the latest checkpoint stored in Google Drive (`t2s_outputs/checkpoints`), and evaluation incrementally appends predicted queries after every batch.

In [ ]:
# Cell 0: GPU Verification
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
gpu_available = torch.cuda.is_available()
print(f"CUDA Available: {gpu_available}")

if not gpu_available:
    raise RuntimeError("ERROR: A CUDA-enabled GPU (e.g. T4) is required for Phase 1 execution. Please change Runtime type to GPU.")

gpu_name = torch.cuda.get_device_name(0)
print(f"Active GPU Device: {gpu_name}")
!nvidia-smi


In [ ]:
# Cell 1: Mount Google Drive & Setup Workspace
import os
from pathlib import Path

if not Path('/content/drive/MyDrive').exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"Drive mount note: {e}")

drive_bundle = Path('/content/drive/MyDrive/phase1_bundle.zip')
if not drive_bundle.exists():
    alt_bundle = Path('/content/phase1_bundle.zip')
    if alt_bundle.exists():
        drive_bundle = alt_bundle
    else:
        raise FileNotFoundError("phase1_bundle.zip not found. Please upload to Drive (MyDrive/phase1_bundle.zip) or use the Colab Files sidebar.")

proj_dir = Path('/content/proj')
proj_dir.mkdir(parents=True, exist_ok=True)
%cd /content/proj

!unzip -o "{drive_bundle}" -d /content/proj

drive_output_dir = Path('/content/drive/MyDrive/t2s_outputs')
drive_output_dir.mkdir(parents=True, exist_ok=True)
print(f"Workspace initialized at {proj_dir}. Mirroring outputs to {drive_output_dir}.")


In [ ]:
# Cell 2: Install Pinned Dependencies
!pip install -q transformers peft trl bitsandbytes accelerate datasets sqlglot psutil matplotlib llama-cpp-python gguf
print("Dependencies installed successfully.")


In [ ]:
# Cell 3: Pre-flight Verification & Leakage Check
!python scripts/check_leakage.py
!pytest -q tests


In [ ]:
# Cell 4: Train QLoRA Model (Resumable)
import json
import time
from pathlib import Path

import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

from src.t2s.train.train_qlora import collate_masked_prompt_completion, format_dataset_samples

ckpt_dir = Path("/content/drive/MyDrive/t2s_outputs/checkpoints")
ckpt_dir.mkdir(parents=True, exist_ok=True)

model_id = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

train_data = format_dataset_samples("data/processed/train.jsonl")
val_data = format_dataset_samples("data/processed/val.jsonl")
print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

out_adapter_dir = Path("models/qlora_adapter")
out_adapter_dir.mkdir(parents=True, exist_ok=True)

t_args = TrainingArguments(
    output_dir=str(ckpt_dir),
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_steps=30,
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=t_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=lambda b: collate_masked_prompt_completion(b, tokenizer, 1024),
)

t0 = time.time()
last_ckpt = None
ckpts = sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
if ckpts:
    last_ckpt = str(ckpts[-1])
    print(f"Resuming training from checkpoint: {last_ckpt}")

trainer.train(resume_from_checkpoint=last_ckpt)
train_time_sec = round(time.time() - t0, 2)

trainer.save_model(str(out_adapter_dir))
tokenizer.save_pretrained(str(out_adapter_dir))
print(f"Saved adapter to {out_adapter_dir}")

# Save training config & stats
results_train_dir = Path("results/training")
results_train_dir.mkdir(parents=True, exist_ok=True)
training_info = {
    "gpu": torch.cuda.get_device_name(0),
    "training_time_sec": train_time_sec,
    "epochs": 2,
    "seed": 42,
    "train_examples": len(train_data),
    "val_examples": len(val_data),
    "final_loss": trainer.state.log_history[-1].get("loss", 0.0)
    if trainer.state.log_history
    else 0.0,
}
with open(results_train_dir / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_info, f, indent=2)

In [ ]:
# Cell 5: Merge QLoRA Adapter Into Base Weights (FP16)
!python scripts/merge_adapter.py --base-model Qwen/Qwen2.5-Coder-3B-Instruct --adapter-dir models/qlora_adapter --output-dir models/merged_model


In [ ]:
# Cell 6: Build llama.cpp and Export GGUF Q4_K_M
!git clone https://github.com/ggerganov/llama.cpp.git /tmp/llama.cpp
%cd /tmp/llama.cpp
!mkdir -p build && cd build && cmake .. -DLLAMA_BUILD_EXAMPLES=ON -DLLAMA_BUILD_TESTS=OFF && cmake --build . --config Release --target llama-quantize -j 2
%cd /content/proj
!python /tmp/llama.cpp/convert_hf_to_gguf.py models/merged_model --outtype f16 --outfile models/merged_f16.gguf
!/tmp/llama.cpp/build/bin/llama-quantize models/merged_f16.gguf models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf Q4_K_M
import os
if os.path.exists('models/merged_f16.gguf'):
    os.remove('models/merged_f16.gguf')
    print('Removed temporary FP16 GGUF file to free disk space.')

# Verify loading
from llama_cpp import Llama
llm = Llama(model_path='models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf', n_gpu_layers=-1, verbose=False)
out = llm('SELECT', max_tokens=1)
print('GGUF Test Generation Token:', out['choices'][0]['text'])


In [ ]:
# Cell 7: Register Model Artifacts
!python scripts/register_artifacts.py


In [ ]:
# Cell 8: Evaluate Configs A, B, C (Transformers GPU Backend)
!python scripts/eval_spider.py --backend transformers --model-path Qwen/Qwen2.5-Coder-3B-Instruct --config A
!python scripts/eval_spider.py --backend transformers --model-path Qwen/Qwen2.5-Coder-3B-Instruct --config B
!python scripts/eval_spider.py --backend transformers --model-path models/merged_model --config C


In [ ]:
# Cell 9: Evaluate Config D (GGUF Q4_K_M)
!python scripts/eval_spider.py --backend openai_compat --model-path models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf --config D


In [ ]:
# Cell 10: Benchmark Latency & Memory (Config D)
!python scripts/bench_latency.py --backend openai_compat --model-path models/Qwen2.5-Coder-3B-Instruct-Q4_K_M.gguf --limit 200


In [ ]:
# Cell 11: Merge Results, Generate Figures & Run Verification Checks
!python scripts/merge_eval_results.py
!python scripts/make_figures.py
!python scripts/make_eval_doc.py
!python scripts/check_phase1.py


In [ ]:
# Cell 12: Bundle Phase 1 Results & Copy to Drive
import zipfile
from pathlib import Path

out_zip = Path('phase1_results.zip')
if out_zip.exists():
    out_zip.unlink()

print('Bundling phase1_results.zip...')
with zipfile.ZipFile(out_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in Path('results').rglob('*'):
        if p.is_file() and 'smoke' not in p.parts:
            zf.write(p, arcname=p.as_posix())
    for p in Path('models/qlora_adapter').rglob('*'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())
    for p in Path('models').glob('*.gguf'):
        if p.is_file():
            zf.write(p, arcname=p.as_posix())

target_drive_zip = Path('/content/drive/MyDrive/phase1_results.zip')
with open(out_zip, 'rb') as f_src, open(target_drive_zip, 'wb') as f_dst:
    f_dst.write(f_src.read())

size_mb = out_zip.stat().st_size / (1024 * 1024)
print(f'Successfully created phase1_results.zip ({size_mb:.2f} MB).')
print(f'Copied to Google Drive: {target_drive_zip}')
